# Experiment 05: Subword Tokenization and POS Tagging

- **Course:** Computational Linguistics and Natural Language Processing (CLNLP) Lab  
- **Date:** 2026-09-03  
- **Student Name:** Mradul Lakhotiya  
- **Roll Number:** [Roll Number]  

---

## 1. Aim & Objectives
- **Aim:** To implement advanced subword-level tokenization techniques using Byte Pair Encoding (BPE) and SentencePiece, and to apply Part-of-Speech (POS) tagging on contextual text using modern NLP libraries.
- **Objectives:**
  1. Implement subword tokenization using a BPE-based tokenizer, using both a pretrained model and training one from scratch.
  2. Implement subword tokenization using SentencePiece, using both a pretrained model and training one from scratch.
  3. Perform POS tagging on a sample sentence using spaCy and NLTK.
  4. Perform POS tagging on a large corpus using spaCy and calculate term frequencies with descriptions.

## 2. Environment Setup & Imports
*(Note: You may need to run `!pip install transformers tokenizers sentencepiece spacy nltk pandas`)*

In [6]:
import os
import pandas as pd
import nltk
import spacy
import sentencepiece as spm
from collections import Counter
from transformers import AutoTokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Ensure datasets
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('tagsets')

try:
    nlp = spacy.load('en_core_web_sm')
except:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

print("Setup Complete.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mradu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mradu\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to
[nltk_data]     C:\Users\mradu\AppData\Roaming\nltk_data...
[nltk_data]   Package tagsets is already up-to-date!


Setup Complete.


## 3. Dataset / Corpus Input

In [7]:
with open('data/input_text.txt', 'r', encoding='utf-8') as f:
    corpus_text = f.read()

sample_sentence = "The young student is reading an interesting book in the library."

print("Corpus loaded successfully.")

Corpus loaded successfully.


## 4. Core Implementation & Results

### Experiment 5.1: BPE Tokenization

In [8]:
print("=== A. Using Pretrained Model (GPT-2 uses BPE) ===")
pretrained_bpe = AutoTokenizer.from_pretrained('gpt2')
tokens_pre_bpe = pretrained_bpe.tokenize(sample_sentence)
ids_pre_bpe = pretrained_bpe.convert_tokens_to_ids(tokens_pre_bpe)
print("Tokens:", tokens_pre_bpe)
print("IDs:   ", ids_pre_bpe)

print("\n=== B. Without Pretrained Model (Train from scratch) ===")
# Initialize tokenizer
tokenizer_bpe = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer_bpe.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"], vocab_size=500)
# Train on the provided corpus file
tokenizer_bpe.train(files=["data/input_text.txt"], trainer=trainer)
output_bpe = tokenizer_bpe.encode(sample_sentence)
print("Tokens:", output_bpe.tokens)
print("IDs:   ", output_bpe.ids)

=== A. Using Pretrained Model (GPT-2 uses BPE) ===


c:\Users\mradu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mradu\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokens: ['The', 'Ġyoung', 'Ġstudent', 'Ġis', 'Ġreading', 'Ġan', 'Ġinteresting', 'Ġbook', 'Ġin', 'Ġthe', 'Ġlibrary', '.']
IDs:    [464, 1862, 3710, 318, 3555, 281, 3499, 1492, 287, 262, 5888, 13]

=== B. Without Pretrained Model (Train from scratch) ===
Tokens: ['The', 'y', 'o', 'un', 'g', 'st', 'udent', 'is', 're', 'ad', 'ing', 'an', 'in', 'ter', 'es', 't', 'ing', 'b', 'o', 'o', 'k', 'in', 'the', 'li', 'b', 'r', 'ar', 'y', '.']
IDs:    [160, 48, 38, 76, 30, 92, 407, 82, 64, 193, 61, 54, 51, 90, 57, 43, 61, 25, 38, 38, 34, 51, 68, 383, 25, 41, 59, 48, 7]


### Experiment 5.2: SentencePiece Tokenization

In [9]:
print("=== A. Using Pretrained Model (T5 uses SentencePiece) ===")
pretrained_spm = AutoTokenizer.from_pretrained('t5-small')
tokens_pre_spm = pretrained_spm.tokenize(sample_sentence)
ids_pre_spm = pretrained_spm.convert_tokens_to_ids(tokens_pre_spm)
print("Tokens:", tokens_pre_spm)
print("IDs:   ", ids_pre_spm)

print("\n=== B. Without Pretrained Model (Train from scratch) ===")
# Train native sentencepiece model
spm.SentencePieceTrainer.train('--input=data/input_text.txt --model_prefix=m --vocab_size=300 --model_type=unigram')
sp = spm.SentencePieceProcessor()
sp.load('m.model')
tokens_sp = sp.encode_as_pieces(sample_sentence)
ids_sp = sp.encode_as_ids(sample_sentence)
print("Tokens:", tokens_sp)
print("IDs:   ", ids_sp)

=== A. Using Pretrained Model (T5 uses SentencePiece) ===


c:\Users\mradu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mradu\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokens: ['▁The', '▁young', '▁student', '▁is', '▁reading', '▁an', '▁interesting', '▁book', '▁in', '▁the', '▁library', '.']
IDs:    [37, 1021, 1236, 19, 1183, 46, 1477, 484, 16, 8, 3595, 5]

=== B. Without Pretrained Model (Train from scratch) ===
Tokens: ['▁The', '▁', 'y', 'o', 'un', 'g', '▁', 's', 't', 'u', 'd', 'en', 't', '▁is', '▁re', 'a', 'd', 'ing', '▁an', '▁in', 'te', 'r', 'es', 't', 'ing', '▁', 'bo', 'ok', '▁in', '▁the', '▁', 'l', 'i', 'b', 'r', 'ar', 'y', '.']
IDs:    [37, 3, 35, 23, 90, 298, 3, 6, 20, 41, 29, 118, 20, 22, 67, 27, 29, 10, 119, 40, 57, 25, 44, 20, 10, 3, 150, 264, 40, 7, 3, 16, 12, 71, 25, 75, 35, 5]


### Experiment 5.3: POS Tagging on Sample Text

In [10]:
print("=== A. POS Tagging Using spaCy ===")
doc = nlp(sample_sentence)
spacy_results = []
for token in doc:
    spacy_results.append((token.text, token.pos_, spacy.explain(token.pos_)))
display(pd.DataFrame(spacy_results, columns=['Token', 'POS Tag', 'Description']))

print("\n=== B. POS Tagging Using NLTK ===")
nltk_tokens = nltk.word_tokenize(sample_sentence)
nltk_tags = nltk.pos_tag(nltk_tokens)
# Note: NLTK uses Penn Treebank tags.
nltk_results = []
for token, tag in nltk_tags:
    nltk_results.append((token, tag, "(Penn Treebank tag)"))
display(pd.DataFrame(nltk_results, columns=['Token', 'POS Tag', 'Description']))

=== A. POS Tagging Using spaCy ===


,Token,POS Tag,Description
0,The,DET,determiner
1,young,ADJ,adjective
2,student,NOUN,noun
3,is,AUX,auxiliary
4,reading,VERB,verb
5,an,DET,determiner
6,interesting,ADJ,adjective
7,book,NOUN,noun
8,in,ADP,adposition
9,the,DET,determiner



=== B. POS Tagging Using NLTK ===


,Token,POS Tag,Description
0,The,DT,(Penn Treebank tag)
1,young,JJ,(Penn Treebank tag)
2,student,NN,(Penn Treebank tag)
3,is,VBZ,(Penn Treebank tag)
4,reading,VBG,(Penn Treebank tag)
5,an,DT,(Penn Treebank tag)
6,interesting,JJ,(Penn Treebank tag)
7,book,NN,(Penn Treebank tag)
8,in,IN,(Penn Treebank tag)
9,the,DT,(Penn Treebank tag)


### Experiment 5.4: POS Tagging on Corpus (Frequency Analysis)

In [11]:
doc_full = nlp(corpus_text)

token_pos_freq = Counter()
for token in doc_full:
    if token.is_alpha:  # Ignoring punctuation and whitespace for clean output
        # Key is a tuple of (Token Text, POS Tag, POS Description)
        token_pos_freq[(token.text.lower(), token.pos_, spacy.explain(token.pos_))] += 1

freq_results = []
for (tok, pos, desc), freq in token_pos_freq.items():
    freq_results.append({
        'Unique Token': tok,
        'POS Tag': pos,
        'Description': desc,
        'Frequency': freq
    })

df_freq = pd.DataFrame(freq_results).sort_values(by='Frequency', ascending=False)
display(df_freq.head(15))  # Display top 15 most frequent tokens

,Unique Token,POS Tag,Description,Frequency
30,the,DET,determiner,47
4,a,DET,determiner,28
15,and,CCONJ,coordinating conjunction,27
82,can,AUX,auxiliary,23
6,of,ADP,adposition,22
44,be,AUX,auxiliary,15
3,is,AUX,auxiliary,14
73,as,ADP,adposition,13
95,bpe,PROPN,proper noun,13
72,word,NOUN,noun,12


## 5. Conclusion & Key Findings
- **Summary:** We observed the dynamic behavior of subword tokenization (BPE and SentencePiece). When using a small, freshly trained corpus, words are highly fragmented. Pretrained models gracefully fragment only complex morphemes. POS tagging cleanly highlighted structural roles in texts, utilizing both spaCy and NLTK logic.
- **Linguistic Insights:** NLTK exposes the granular Penn Treebank tag set (e.g., `VBG` for verb gerunds) whereas spaCy Universal POS defaults to higher-level categorizations (`VERB`). Subword tokenizers automatically parse affixes like `-ing` gracefully without overt grammatical rules.